# Install libs

In [ ]:
!pip install vibdata==1.1.1 signalAI==0.0.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.1 MB/s eta 0:00:00


# Import Libs

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Basic imports
import numpy as np
import numpy.typing as npt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy

# vibdata
import vibdata.raw as raw_datasets
from vibdata.deep.DeepDataset import DeepDataset, convertDataset
from vibdata.deep.signal.transforms import (
    Sequential,
    SplitSampleRate,
    FeatureExtractor,
    FilterByValue,
    Split
)
from vibdata.deep.signal.core import SignalSample

# SignalAI
from signalAI.experiments.features_1d import Features1DExperiment
from signalAI.utils.group_dataset import GroupDataset
from signalAI.utils.fold_idx_generator import (
    FoldIdxGeneratorUnbiased,
    FoldIdxGeneratorBiased,
)

class GroupCWRULoad(GroupDataset):
    @staticmethod
    def _assigne_group(sample: SignalSample) -> int:
        return sample["metainfo"]["load"]

# Deep Learning Experiments

## Import CRWU dataset

In [ ]:
raw_root_dir = "../data/raw_data/cwru"
raw_dataset = raw_datasets.CWRU_raw(raw_root_dir, download=True)

Cached downloading...
Hash: md5:d7d3042161080fc82e99d78464fa2914
From (original): https://drive.google.com/uc?id=1G2vfms1QDlkdzqL_LAQdMIQAoludxBNj
From (redirected): https://drive.google.com/uc?id=1G2vfms1QDlkdzqL_LAQdMIQAoludxBNj&confirm=t&uuid=b7b0f967-d8eb-4554-9ae9-d03576cae7e2
To: ../data/raw_data/cwru/CWRU_raw/CWRU.zip
100%|██████████| 245M/245M [00:02<00:00, 116MB/s]


## Time domain

### Filter by 12k SampleRate

In [ ]:
transforms_time = Sequential(
    [
        SplitSampleRate()
    ]
)
print(transforms_time)

Sequential(transforms=[SplitSampleRate()])


In [ ]:
deep_root_dir_time = "../data/deep_data/deep_learning"
deep_dataset_time = convertDataset(raw_dataset,filter=FilterByValue(on_field="sample_rate", values=12000),transforms=transforms_time, dir_path=deep_root_dir_time, batch_size=32)

Transformando


Converting CWRU: 100%|██████████| 10/10 [00:01<00:00,  5.14it/s]


## Generate Unbiased Folds Single Round

In [ ]:
# folds_singleround_deep = FoldIdxGeneratorUnbiased(deep_dataset_time, GroupCWRULoad , dataset_name="CWRU12k_single").generate_folds()
# folds_singleround_deep

## Generate Unbiased Folds MultiRound

In [ ]:
class GroupMultiRoundCWRULoad(GroupDataset):
    @staticmethod
    def _assigne_group(sample: SignalSample) -> int:
        sample_metainfo = sample["metainfo"]
        return sample_metainfo["label"].astype(str) + " " + sample_metainfo["load"].astype(int).astype(str)

CLASS_DEF = {0: "N", 1: "O", 2: "I", 3: "R"}
CONDITION_DEF = {"0": "0", "1": "1", "2": "2", "3": "3"}
folds_multiround_deep = FoldIdxGeneratorUnbiased(deep_dataset_time, GroupMultiRoundCWRULoad, dataset_name="CWRU12k_multi", multiround=True, class_def=CLASS_DEF, condition_def=CONDITION_DEF).generate_folds()
folds_multiround_deep

Grouping dataset: 100%|██████████| 2520/2520 [00:00<00:00, 6402.23sample/s]


Per round splits:  4
Number of repeats:  8
Total combinations of folds: 256
Total combinations between folds 174792640
Time to generate combinations: 24.15 seconds


  0%|          | 125877/172869516 [00:03<1:28:43, 32446.47it/s]

Total combs:  8
round:  0
fold:  0 -> N 0, O 0, I 0, R 0,  => 0
fold:  1 -> N 1, O 1, I 1, R 1,  => 1
fold:  2 -> N 2, O 2, I 2, R 2,  => 2
fold:  3 -> N 3, O 3, I 3, R 3,  => 3

round:  1
fold:  0 -> N 0, O 3, I 2, R 2,  => 4
fold:  1 -> N 1, O 2, I 1, R 3,  => 5
fold:  2 -> N 2, O 1, I 0, R 1,  => 6
fold:  3 -> N 3, O 0, I 3, R 0,  => 7

round:  2
fold:  0 -> N 0, O 2, I 2, R 3,  => 8
fold:  1 -> N 1, O 1, I 3, R 2,  => 9
fold:  2 -> N 2, O 3, I 0, R 0,  => 10
fold:  3 -> N 3, O 0, I 1, R 1,  => 11

round:  3
fold:  0 -> N 0, O 2, I 3, R 2,  => 12
fold:  1 -> N 1, O 3, I 0, R 1,  => 13
fold:  2 -> N 2, O 0, I 1, R 0,  => 14
fold:  3 -> N 3, O 1, I 2, R 3,  => 15

round:  4
fold:  0 -> N 0, O 1, I 3, R 3,  => 16
fold:  1 -> N 1, O 0, I 0, R 2,  => 17
fold:  2 -> N 2, O 2, I 2, R 0,  => 18
fold:  3 -> N 3, O 3, I 1, R 1,  => 19

round:  5
fold:  0 -> N 0, O 3, I 1, R 1,  => 20
fold:  1 -> N 1, O 0, I 2, R 3,  => 21
fold:  2 -> N 2, O 2, I 0, R 0,  => 22
fold:  3 -> N 3, O 1, I 3, R 2, 

[array([0, 0, 0, ..., 3, 3, 3]),
 array([0, 0, 0, ..., 1, 1, 1]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([0, 0, 0, ..., 3, 3, 3]),
 array([0, 0, 0, ..., 0, 0, 0]),
 array([0, 0, 0, ..., 1, 1, 1]),
 array([0, 0, 0, ..., 2, 2, 2]),
 array([0, 0, 0, ..., 0, 0, 0])]

## DeepLearning Experiments

### Utils

In [ ]:
# vibclassifier/experiments/base.py
from abc import ABC, abstractmethod
import json
from typing import Optional, Dict, Any
from vibdata.raw.base import RawVibrationDataset
from vibdata.deep.signal.transforms import Transform

class Experiment(ABC):
    """Classe base abstrata para todos os experimentos de classificação de vibração."""

    def __init__(
        self,
        name: str,
        description: str,
        dataset: Optional[RawVibrationDataset] = None,
        data_transform: Optional[Transform] = None,
        feature_selector = None,
        model = None
    ):
        """
        Inicializa o experimento.

        Args:
            name: Nome identificador do experimento
            description: Descrição detalhada do experimento
            dataset: Conjunto de dados de vibração
            data_transform: Transformação a ser aplicada nos dados brutos
            data_division_method: Método de divisão dos dados (e.g., 'kfold', 'holdout')
            data_division_params: Parâmetros para o método de divisão
            feature_selector: Seletor de features (para experimentos com extração)
            model: Modelo de machine learning/deep learning
        """
        self.name = name
        self.description = description
        self.dataset = dataset
        self.data_transform = data_transform
        self.feature_selector = feature_selector
        self.model = model

        # Resultados serão armazenados aqui
        self.results = {}

    @abstractmethod
    def prepare_data(self):
        """Prepara os dados para o experimento."""
        pass

    @abstractmethod
    def run(self):
        """Executa o experimento completo."""
        pass

    def save_results(self, filepath: str):
        """Salva os resultados do experimento."""
        # Implementação básica - pode ser extendida
        with open(filepath, 'w') as f:
            json.dump(self.results, f)

    def load_results(self, filepath: str):
        """Carrega resultados de um experimento anterior."""
        with open(filepath, 'r') as f:
            self.results = json.load(f)

    def __str__(self):
        return f"Experiment: {self.name}\nDescription: {self.description}"

In [ ]:
# vibclassifier/experiments/deep_torch.py
import os
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import numpy as np
from typing import List, Dict, Optional, Tuple, Union
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from signalAI.utils.metrics import calculate_metrics
from signalAI.utils.experiment_result import ExperimentResults, FoldResults
import copy

class TorchVibrationDataset(Dataset):
    """Wrapper to convert dataset samples into Torch tensors."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Função auxiliar KL
def kl_divergence(rho, rho_hat):
    rho_hat = torch.mean(rho_hat, dim=0)
    rho = torch.tensor([rho] * len(rho_hat), device=rho_hat.device)
    epsilon = 1e-7
    term1 = rho * torch.log((rho + epsilon) / (rho_hat + epsilon))
    term2 = (1 - rho) * torch.log((1 - rho + epsilon) / (1 - rho_hat + epsilon))
    return torch.sum(term1 + term2)

class DeepLearningExperiment(Experiment):
    def __init__(
        self,
        name: str,
        description: str,
        dataset,
        data_fold_idxs: List[int],
        model: nn.Module,
        criterion: Optional[nn.Module] = None,
        # Parâmetros adaptados para o autoencoder
        reconstruction_criterion: Optional[nn.Module] = None,
        recon_loss_weight: float = 1.0,
        sparsity_target: Optional[float] = None,
        sparsity_weight: float = 0.0,
        pretrain_epochs: int = 0, # Épocas de treinamento do autoencoder
        optimizer_class: Optional[torch.optim.Optimizer] = optim.Adam,
        batch_size: int = 32,
        lr: float = 1e-3,
        num_epochs: int = 20, # Épocas de treino do classificador
        val_split: float = 0.2,
        output_dir: str = "results_torch",
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        **kwargs
    ):
        super().__init__(name, description, dataset, model=model, **kwargs)
        self.data_fold_idxs = data_fold_idxs
        self.output_dir = Path(output_dir)
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.pretrain_epochs = pretrain_epochs
        self.val_split = val_split
        self.device = device
        self.optimizer_class = optimizer_class
        self.lr = lr
        self.criterion = criterion if criterion is not None else nn.CrossEntropyLoss()

        self.reconstruction_criterion = reconstruction_criterion
        self.recon_loss_weight = recon_loss_weight
        self.sparsity_target = sparsity_target
        self.sparsity_weight = sparsity_weight

        self.is_sae_task = self.sparsity_target is not None and self.sparsity_weight > 0.0
        # Define tipo do AutoEncoder utilizado
        self.is_autoencoder_task = reconstruction_criterion is not None or self.is_sae_task or "AE1D" in model.__class__.__name__

        if self.is_sae_task and self.reconstruction_criterion is None:
             print("Warning: SAE task detected but no reconstruction_criterion. Defaulting to MSELoss.")
             self.reconstruction_criterion = nn.MSELoss()

        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs")
            model = torch.nn.DataParallel(model)

        # Guarda encoder e decoder
        self.original_model = model.module if isinstance(model, nn.DataParallel) else model

        self.n_outer_folds = len(np.unique(data_fold_idxs))
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.prepare_data()

    def prepare_data(self):
        features, labels = [], []
        for sample in self.dataset:
            features.append(sample['signal'][0])
            labels.append(sample['metainfo']['label'])
        self.X = np.array(features)
        self.label_encoder = LabelEncoder()
        self.y = self.label_encoder.fit_transform(labels)

    def _train_one_fold(
        self, X_train, y_train, X_test, y_test, fold_idx: int
    ) -> FoldResults:

        train_dataset = TorchVibrationDataset(X_train, y_train)
        test_dataset = TorchVibrationDataset(X_test, y_test)
        val_size = int(self.val_split * len(train_dataset))
        train_size = len(train_dataset) - val_size
        train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        model = copy.deepcopy(self.model.to(self.device))
        model_core = model.module if isinstance(model, nn.DataParallel) else model

        # Verificação da estrutura de AE (encoder, decoder, classifier)
        has_ae_structure = hasattr(model_core, 'encoder') and hasattr(model_core, 'decoder') and hasattr(model_core, 'classifier')

        # Treinamento do AutoEncoder
        if self.is_autoencoder_task and self.pretrain_epochs > 0 and has_ae_structure:
            print(f"[Fold {fold_idx}] AutoEncoder training ({self.pretrain_epochs} epochs)...")

            # Otimizador Encoder + Decoder
            optimizer_ae = self.optimizer_class([
                {'params': model_core.encoder.parameters()},
                {'params': model_core.decoder.parameters()}
            ], lr=self.lr)

            for epoch in range(self.pretrain_epochs):
                model.train()
                running_recon_loss = 0.0

                for xb, _ in train_loader:
                    xb = xb.to(self.device)
                    input_data = xb

                    if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                        xb = xb.unsqueeze(1)
                    elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                         side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                    optimizer_ae.zero_grad()
                    outputs = model(xb) # (class, recon, [sparsity])

                    if isinstance(outputs, tuple):
                        # Foco na reconstrução
                        reconstruction = outputs[1]

                        loss = self.reconstruction_criterion(reconstruction, input_data)

                        # Adiciona esparsidade se for SAE
                        if self.is_sae_task and len(outputs) > 2:
                            latent_features = outputs[2]
                            loss += self.sparsity_weight * kl_divergence(self.sparsity_target, latent_features)

                        loss.backward()
                        optimizer_ae.step()
                        running_recon_loss += loss.item() * input_data.size(0)

                avg_recon_loss = running_recon_loss / len(train_loader.dataset)
                if (epoch + 1) % 5 == 0 or epoch == 0:
                    print(f"  [Pre-train] Epoch {epoch+1}/{self.pretrain_epochs} Recon Loss: {avg_recon_loss:.4f}")

        # Treino do classificador
        print(f"[Fold {fold_idx}] Classifier training ({self.num_epochs} epochs)...")

        # Define otimizador para a fase supervisionada
        if has_ae_structure and self.is_autoencoder_task:
            # Se for AE: Treina Encoder + Classifier (Decoder congelado ou ignorado pelo otimizador)
            optimizer_clf = self.optimizer_class([
                {'params': model_core.encoder.parameters()},
                {'params': model_core.classifier.parameters()}
            ], lr=self.lr)
        else:
            # Se for MLP/CNN padrão: Treina todos os parâmetros
            optimizer_clf = self.optimizer_class(model.parameters(), lr=self.lr)

        train_losses, val_losses = [], []

        for epoch in range(self.num_epochs):
            epoch_start = time.time()
            model.train()
            running_loss = 0.0

            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)

                # Ajuste de shape
                if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                     xb = xb.unsqueeze(1)
                elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                     side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                optimizer_clf.zero_grad()
                outputs = model(xb)

                # Cálculo da perda apenas de CLASSIFICAÇÃO
                if isinstance(outputs, tuple):
                    classification_output = outputs[0] # Pega apenas a classificação
                else:
                    classification_output = outputs # Modelo padrão

                loss = self.criterion(classification_output, yb)

                loss.backward()
                optimizer_clf.step()
                running_loss += loss.item() * xb.size(0)

            avg_train_loss = running_loss / len(train_loader.dataset)

            # Validação
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    # Ajuste de shape
                    if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                         xb = xb.unsqueeze(1)
                    elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                         side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                    outputs = model(xb)

                    if isinstance(outputs, tuple):
                        classification_output = outputs[0]
                    else:
                        classification_output = outputs

                    loss = self.criterion(classification_output, yb)
                    val_loss += loss.item() * xb.size(0)

            avg_val_loss = val_loss / len(val_loader.dataset)

            train_losses.append(avg_train_loss)
            val_losses.append(avg_val_loss)

            epoch_time = time.time() - epoch_start
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [Supervised] Epoch {epoch+1}/{self.num_epochs} Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Time: {epoch_time:.2f}s")

        plt.figure()
        plt.plot(train_losses, label="Train Loss (Clf)")
        plt.plot(val_losses, label="Val Loss (Clf)")
        plt.legend(); plt.title(f"Loss Curve - Fold {fold_idx}")
        plt.savefig(os.path.join(self.dir_path, f"loss_curve_fold{fold_idx}_{self.start_time}.png")); plt.close()

        torch.save(model_core.state_dict(), os.path.join(self.dir_path, f"model_fold{fold_idx}.pt"))

        # Teste
        y_true, y_pred, y_proba = [], [], []
        model.eval()
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                if any(isinstance(m, nn.Conv1d) for m in model.modules()) and xb.ndim == 2:
                     xb = xb.unsqueeze(1)
                elif any(isinstance(m, nn.Conv2d) for m in model.modules()) and xb.ndim == 2:
                     side = int(np.sqrt(xb.shape[1])); xb = xb.view(xb.size(0), 1, side, side)

                outputs = model(xb)
                if isinstance(outputs, tuple):
                    classification_output = outputs[0]
                else:
                    classification_output = outputs

                probs = torch.softmax(classification_output, dim=1)
                preds = torch.argmax(probs, dim=1)
                y_true.extend(yb.cpu().numpy()); y_pred.extend(preds.cpu().numpy()); y_proba.extend(probs.cpu().numpy())

        metrics = calculate_metrics(np.array(y_true), np.array(y_pred), np.array(y_proba))
        return FoldResults(fold_idx, np.array(y_true), np.array(y_pred), np.array(y_proba), metrics)

    def run(self) -> ExperimentResults:
        self.start_time = time.strftime("%Y%m%d_%H%M%S")
        self.dir_path = os.path.join(self.output_dir, f"results_{self.name}_{self.start_time}")
        os.makedirs(self.dir_path, exist_ok=True)

        results = ExperimentResults(
            experiment_name=self.name, description=self.description,
            model_name=self.original_model.__class__.__name__, feature_names=None,
            config={'n_outer_folds': self.n_outer_folds, 'pretrain_epochs': self.pretrain_epochs,
                    'finetune_epochs': self.num_epochs, 'batch_size': self.batch_size, 'lr': self.lr}
        )

        for outer_fold in range(self.n_outer_folds):
            print(f"\n=== Outer Fold {outer_fold+1}/{self.n_outer_folds} ===")
            train_mask = self.data_fold_idxs != outer_fold
            test_mask = self.data_fold_idxs == outer_fold

            try:
                fold_result = self._train_one_fold(self.X[train_mask], self.y[train_mask], self.X[test_mask], self.y[test_mask], outer_fold)
                results.add_fold_result(fold_result)
                print(f"  Result: Acc={fold_result.metrics['accuracy']:.4f}, F1={fold_result.metrics['f1']:.4f}")
            except Exception as e:
                print(f"Error in fold {outer_fold}: {e}")
                import traceback; traceback.print_exc()

        results.calculate_overall_metrics()
        results.save_json(os.path.join(self.dir_path, f"results.json"))
        print("\n=== Final Results ===")
        print(f"Mean Accuracy: {results.overall_metrics['accuracy']:.4f}")
        return results

### 1D MLP adaptado para 12k

In [ ]:
class MLP1D_12k(nn.Module):
    def __init__(self, input_length: int = 12000, num_classes: int = 4):
        super().__init__()

        # Camada adicional para adaptar a entrada de 12000 para 1024 progressivamente
        self.adaptation_layers = nn.Sequential(
            # 12000 -> 4096
            nn.Linear(input_length, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.7), # Dropout alto para evitar overfitting na entrada

            # 4096 -> 2048
            nn.Linear(4096, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            # 2048 -> 1024 (Conecta com a arquitetura original)
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4)
        )

        # Arquitetura original:
        self.fc3 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True)
        )

        self.fc4 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True)
        )

        self.fc5 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True)
        )

        self.fc6 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True)
        )

        self.fc7 = nn.Sequential(
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        out = torch.flatten(x, 1)

        # Passa pela adaptação primeiro
        out = self.adaptation_layers(out)

        # Segue o fluxo normal
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.fc5(out)
        out = self.fc6(out)
        out = self.fc7(out)

        return out

In [ ]:
import copy

# Configuração Inicial do Modelo Base (Template)
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
print(f"Total de Rounds para executar: {len(folds_multiround_deep)}")

# Instancia o modelo "limpo" que será copiado a cada rodada
base_model = MLP1D_12k(input_length=input_length, num_classes=num_classes)

# Lista para armazenar os resultados de cada rodada
multiround_results = []
accuracies = []
f1_scores = []

# Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy garante que os pesos sejam resetados a cada rodada
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"mlp1d_12k_round_{round_idx}",
        description=f"MLP 1D - Multiround execution (Round {round_idx})",
        dataset=deep_dataset_time,

        # Passado apenas o array de folds da rodada atual
        data_fold_idxs=current_folds,

        model=model_copy,
        batch_size=64,
        lr=3e-4,
        pretrain_epochs=0,
        num_epochs=100,
        output_dir=f"results_multiround/round_{round_idx}"
    )

    # Executa e guarda o resultado
    result = exp.run()
    multiround_results.append(result)

    # Coleta métricas
    accuracies.append(result.overall_metrics['accuracy'])
    f1_scores.append(result.overall_metrics['mean_f1'])

# Consolidação Final dos Resultados
print("\n" + "="*40)
print("RELATÓRIO FINAL MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

print(f"Total Rounds Executados: {len(multiround_results)}")
print(f"Acurácia Média Global: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio Global: {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração ---
Input length: 12000
Num classes: 4
Total de Rounds para executar: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 1.3808, Val Loss: 1.4567, Time: 0.65s
  [Supervised] Epoch 5/100 Train Loss: 0.9408, Val Loss: 1.4219, Time: 0.63s
  [Supervised] Epoch 10/100 Train Loss: 0.3052, Val Loss: 1.2914, Time: 0.63s
  [Supervised] Epoch 15/100 Train Loss: 0.1989, Val Loss: 1.4008, Time: 0.63s
  [Supervised] Epoch 20/100 Train Loss: 0.1529, Val Loss: 1.3737, Time: 0.63s
  [Supervised] Epoch 25/100 Train Loss: 0.1232, Val Loss: 1.4482, Time: 0.63s
  [Supervised] Epoch 30/100 Train Loss: 0.1031, Val Loss: 1.6142, Time: 0.63s
  [Supervised] Epoch 35/100 Train Loss: 0.1048, Val Loss: 1.6158, Time: 0.63s
  [Supervised] Epoch 40/100 Train Loss: 0.0870, Val Loss: 1.6815, Time: 0.63s
  [Supervised] Epoch 45/100 Train Loss: 0.0727, Val Loss: 1.6951, Time: 0.63s
  [Supervised] Epoch 50/100 Train L

In [ ]:
!cp -r "/content/results_multiround" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D AutoEncoder

In [ ]:
import torch
import torch.nn as nn

class AE1D(nn.Module):
    """
    Implementação do Autoencoder 1D Adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent) -> 128 -> 256 -> 512 -> 12000.
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4, dropout_rate: float = 0.4):
        super(AE1D, self).__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            # Camada Latente: 128 -> 64
            nn.Linear(128, latent_dim)
        )

        # --- Decoder ---
        # Simétrico ao encoder
        self.decoder = nn.Sequential(
            # Latente -> 128
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            # 128 -> 256
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            # 256 -> 512
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução Final: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # Classificador (Fine-tuning)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # Flatten para garantir (Batch, 12000)
        x = torch.flatten(x, 1)

        # Encoder
        latent_features = self.encoder(x)

        # Decoder (Reconstrução do sinal de 12k)
        reconstruction = self.decoder(latent_features)

        # Classifier
        classification_output = self.classifier(latent_features)

        return classification_output, reconstruction, latent_features

In [ ]:
# Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração AE Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que você já gerou folds_multiround_deep anteriormente
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios (Definidos uma vez, reutilizados)
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base (será copiado a cada iteração)
base_model = AE1D(input_length=input_length, latent_dim=64, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_AE = []
accuracies_AE = []
f1_scores_AE = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para resetar os pesos do modelo (Encoder e Decoder)
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"ae1d_12k_round_{round_idx}",
        description=f"Autoencoder 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,
        data_fold_idxs=current_folds,
        model=model_copy,

        # Parâmetros Específicos do AE
        reconstruction_criterion=reconstruction_criterion,
        criterion=classification_criterion,
        recon_loss_weight=1.0,

        # Hiperparâmetros de Treino
        pretrain_epochs=100,  # Fase 1: Aprender a reconstruir (não supervisionado)
        num_epochs=100,       # Fase 2: Aprender a classificar (supervisionado)
        batch_size=64,
        lr=3e-4,

        # Organização de saídas
        output_dir=f"results_multiround_AE1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_AE.append(result)

    # Coleta métricas globais desta rodada
    accuracies_AE.append(result.overall_metrics['accuracy'])
    f1_scores_AE.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL AE MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_AE)
std_acc = np.std(accuracies_AE)
mean_f1 = np.mean(f1_scores_AE)
std_f1 = np.std(f1_scores_AE)

print(f"Rounds Executados: {len(multiround_results_AE)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração AE Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 0.2039
  [Pre-train] Epoch 5/100 Recon Loss: 0.1314
  [Pre-train] Epoch 10/100 Recon Loss: 0.1309
  [Pre-train] Epoch 15/100 Recon Loss: 0.1309
  [Pre-train] Epoch 20/100 Recon Loss: 0.1307
  [Pre-train] Epoch 25/100 Recon Loss: 0.1307
  [Pre-train] Epoch 30/100 Recon Loss: 0.1306
  [Pre-train] Epoch 35/100 Recon Loss: 0.1306
  [Pre-train] Epoch 40/100 Recon Loss: 0.1303
  [Pre-train] Epoch 45/100 Recon Loss: 0.1302
  [Pre-train] Epoch 50/100 Recon Loss: 0.1301
  [Pre-train] Epoch 55/100 Recon Loss: 0.1300
  [Pre-train] Epoch 60/100 Recon Loss: 0.1301
  [Pre-train] Epoch 65/100 Recon Loss: 0.1294
  [Pre-train] Epoch 70/100 Recon Loss: 0.1295
  [Pre-train] Epoch 75/100 Recon Loss: 0.1296
  [Pre-train] Epoch 80/100 Recon Loss: 0.1296
  [Pre-train] Epoch 85/100 Recon

In [ ]:
!cp -r "/content/results_multiround_AE1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D Sparse AutoEncoder

In [ ]:
class SAE1D(nn.Module):
    """
    Implementação do Sparse Autoencoder 1D adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent).
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4):
        super(SAE1D, self).__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            # Redução drástica necessária para viabilidade
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2), # Adicionado Dropout leve

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada Latente: 128 -> Latent Dim
            nn.Linear(128, latent_dim)
        )

        # A Sigmoid é OBRIGATÓRIA para SAE se você usar KL Divergence Loss
        # Ela força os neurônios latentes a ficarem entre [0, 1] (probabilidade de ativação)
        self.sparsity_activation = nn.Sigmoid()

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # Classificador (Fine-tuning)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # Garante (Batch, 12000)
        x = torch.flatten(x, 1)

        # 1. Codificação Linear
        features = self.encoder(x)

        # 2. Ativação Esparsa (Latent Space)
        # Importante: A saída aqui estará entre 0 e 1
        latent_features = self.sparsity_activation(features)

        # 3. Reconstrução
        reconstruction = self.decoder(latent_features)

        # 4. Classificação
        classification_output = self.classifier(latent_features)

        return classification_output, reconstruction, latent_features

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração SAE Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base SAE (será copiado a cada iteração)
# Certifique-se de que SAE1D é a versão compatível com 12k (SAE1D_12k) se necessário
base_model = SAE1D(input_length=input_length, latent_dim=64, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_SAE = []
accuracies_SAE = []
f1_scores_SAE = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para resetar os pesos do modelo e garantir independência estatística
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"sae1d_12k_round_{round_idx}",
        description=f"Sparse Autoencoder 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Critérios de Perda
        reconstruction_criterion=reconstruction_criterion,
        criterion=classification_criterion,
        recon_loss_weight=1.0,

        # --- Parâmetros Específicos do SAE ---
        sparsity_target=0.05,  # Rho (5% de ativação)
        sparsity_weight=1.0,   # Beta (Peso da penalidade KL)

        # Hiperparâmetros de Treino
        pretrain_epochs=100,  # Treino não supervisionado (MSE + Esparsidade)
        num_epochs=100,       # Treino supervisionado (CrossEntropy)
        batch_size=64,
        lr=3e-4,

        # Organização de saídas
        output_dir=f"results_multiround_SAE1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_SAE.append(result)

    # Coleta métricas globais desta rodada
    # Nota: Verifique se sua classe retorna 'mean_f1' ou apenas 'f1' no dicionário overall_metrics
    accuracies_SAE.append(result.overall_metrics['accuracy'])
    f1_scores_SAE.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL SAE MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_SAE)
std_acc = np.std(accuracies_SAE)
mean_f1 = np.mean(f1_scores_SAE)
std_f1 = np.std(f1_scores_SAE)

print(f"Rounds Executados: {len(multiround_results_SAE)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração SAE Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 27.8365
  [Pre-train] Epoch 5/100 Recon Loss: 5.4008
  [Pre-train] Epoch 10/100 Recon Loss: 0.4922
  [Pre-train] Epoch 15/100 Recon Loss: 0.1593
  [Pre-train] Epoch 20/100 Recon Loss: 0.1380
  [Pre-train] Epoch 25/100 Recon Loss: 0.1370
  [Pre-train] Epoch 30/100 Recon Loss: 0.1354
  [Pre-train] Epoch 35/100 Recon Loss: 0.1400
  [Pre-train] Epoch 40/100 Recon Loss: 0.1357
  [Pre-train] Epoch 45/100 Recon Loss: 0.1388
  [Pre-train] Epoch 50/100 Recon Loss: 0.1384
  [Pre-train] Epoch 55/100 Recon Loss: 0.1360
  [Pre-train] Epoch 60/100 Recon Loss: 0.1364
  [Pre-train] Epoch 65/100 Recon Loss: 0.1361
  [Pre-train] Epoch 70/100 Recon Loss: 0.1345
  [Pre-train] Epoch 75/100 Recon Loss: 0.1329
  [Pre-train] Epoch 80/100 Recon Loss: 0.1339
  [Pre-train] Epoch 85/100 Rec

In [ ]:
!cp -r "/content/results_multiround_SAE1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D Denoising AutoEncoder

In [ ]:
class DAE1D(nn.Module):
    """
    Implementação do Denoising Autoencoder (DAE) adaptado para 12k pontos.
    Arquitetura: 12000 -> 512 -> 256 -> 128 -> 64 (Latent).
    """
    def __init__(self, input_length: int = 12000, latent_dim: int = 64, num_classes: int = 4, noise_factor: float = 0.5):
        super(DAE1D, self).__init__()
        self.noise_factor = noise_factor

        # Encoder
        self.encoder = nn.Sequential(
            # Camada 1: Compressão Direta (12000 -> 512)
            nn.Linear(input_length, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Camada 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            # Latent
            nn.Linear(128, latent_dim)
        )

        # Decoder
        # Simétrico
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            # Reconstrução: 512 -> 12000
            nn.Linear(512, input_length)
        )

        # --- Classificador ---
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # (Batch, 12000)
        x = torch.flatten(x, 1)

        # Denoising Injection
        if self.training:
            # Adiciona ruído apenas durante o treino
            noise = torch.randn_like(x) * self.noise_factor
            x_noisy = x + noise
        else:
            x_noisy = x

        # Encoder
        latent_features = self.encoder(x_noisy)

        # Decoder
        reconstruction = self.decoder(latent_features)

        # Classifier
        classification_output = self.classifier(latent_features)

        # (Classification, Reconstruction, Latent[Opcional])
        return classification_output, reconstruction, latent_features

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração DAE Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Critérios (Definidos uma vez)
classification_criterion = nn.CrossEntropyLoss()
reconstruction_criterion = nn.MSELoss()

# Instancia o modelo base DAE (será copiado a cada iteração)
# noise_factor=0.5 define a intensidade do ruído gaussiano injetado no treino
base_model = DAE1D(input_length=input_length, latent_dim=64, num_classes=num_classes, noise_factor=0.5)

# Listas para armazenar métricas
multiround_results_DAE = []
accuracies_DAE = []
f1_scores_DAE = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para resetar os pesos do modelo (Encoder e Decoder)
    # Isso garante que o ruído e o aprendizado comecem do zero em cada rodada
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"dae1d_12k_round_{round_idx}",
        description=f"Denoising Autoencoder 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Critérios de Perda
        reconstruction_criterion=reconstruction_criterion, # Ativa a fase de Denoising
        criterion=classification_criterion,                # Ativa a fase de Classificação
        recon_loss_weight=1.0,

        # Hiperparâmetros de Treino
        pretrain_epochs=100,  # Fase 1: Aprender a limpar o ruído (Denoising Task)
        num_epochs=100,       # Fase 2: Aprender a classificar (Diagnostic Task)
        batch_size=64,        # Batch maior ajuda na estabilidade do AE
        lr=3e-4,

        # Organização de saídas
        output_dir=f"results_multiround_DAE1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_DAE.append(result)

    # Coleta métricas globais desta rodada
    accuracies_DAE.append(result.overall_metrics['accuracy'])
    f1_scores_DAE.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL DAE MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_DAE)
std_acc = np.std(accuracies_DAE)
mean_f1 = np.mean(f1_scores_DAE)
std_f1 = np.std(f1_scores_DAE)

print(f"Rounds Executados: {len(multiround_results_DAE)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração DAE Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] AutoEncoder training (100 epochs)...
  [Pre-train] Epoch 1/100 Recon Loss: 0.2263
  [Pre-train] Epoch 5/100 Recon Loss: 0.1339
  [Pre-train] Epoch 10/100 Recon Loss: 0.1329
  [Pre-train] Epoch 15/100 Recon Loss: 0.1328
  [Pre-train] Epoch 20/100 Recon Loss: 0.1329
  [Pre-train] Epoch 25/100 Recon Loss: 0.1322
  [Pre-train] Epoch 30/100 Recon Loss: 0.1317
  [Pre-train] Epoch 35/100 Recon Loss: 0.1312
  [Pre-train] Epoch 40/100 Recon Loss: 0.1295
  [Pre-train] Epoch 45/100 Recon Loss: 0.1280
  [Pre-train] Epoch 50/100 Recon Loss: 0.1274
  [Pre-train] Epoch 55/100 Recon Loss: 0.1253
  [Pre-train] Epoch 60/100 Recon Loss: 0.1232
  [Pre-train] Epoch 65/100 Recon Loss: 0.1209
  [Pre-train] Epoch 70/100 Recon Loss: 0.1193
  [Pre-train] Epoch 75/100 Recon Loss: 0.1153
  [Pre-train] Epoch 80/100 Recon Loss: 0.1132
  [Pre-train] Epoch 85/100 Reco

In [ ]:
!cp -r "/content/results_multiround_DAE1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D CNN

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, input_length: int, num_classes: int):
        """
        1D CNN for vibration signal classification.
        Args:
            input_length: length of the input signal
            num_classes: number of output classes
        """
        super(CNN1D, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.pool3 = nn.AdaptiveMaxPool1d(16)  # reduce dynamically to fixed size

        # compute flattened size
        example_input = torch.zeros(1, 1, input_length)  # [B, C, L]
        with torch.no_grad():
            x = self.pool1(F.relu(self.bn1(self.conv1(example_input))))
            x = self.pool2(F.relu(self.bn2(self.conv2(x))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            flattened_size = x.shape[1] * x.shape[2]

        self.fc1 = nn.Linear(flattened_size, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape: [B, L] or [B, 1, L]
        if x.ndim == 2:
            x = x.unsqueeze(1)  # add channel dim

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração CNN Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base CNN (será copiado a cada iteração)
base_model = CNN1D(input_length=input_length, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_CNN = []
accuracies_CNN = []
f1_scores_CNN = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para resetar os pesos do modelo e garantir independência estatística
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"cnn1d_12k_round_{round_idx}",
        description=f"1D CNN Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino (Supervisionado Padrão)
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # CNN não tem fase de pré-treino não supervisionado

        # Organização de saídas
        output_dir=f"results_multiround_CNN1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_CNN.append(result)

    # Coleta métricas globais desta rodada
    accuracies_CNN.append(result.overall_metrics['accuracy'])
    f1_scores_CNN.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL CNN MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_CNN)
std_acc = np.std(accuracies_CNN)
mean_f1 = np.mean(f1_scores_CNN)
std_f1 = np.std(f1_scores_CNN)

print(f"Rounds Executados: {len(multiround_results_CNN)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração CNN Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 1.3559, Val Loss: 1.2796, Time: 1.38s
  [Supervised] Epoch 5/100 Train Loss: 0.9187, Val Loss: 0.8294, Time: 0.65s
  [Supervised] Epoch 10/100 Train Loss: 0.6130, Val Loss: 0.5430, Time: 0.66s
  [Supervised] Epoch 15/100 Train Loss: 0.4228, Val Loss: 0.3547, Time: 0.66s
  [Supervised] Epoch 20/100 Train Loss: 0.2932, Val Loss: 0.2580, Time: 0.66s
  [Supervised] Epoch 25/100 Train Loss: 0.2081, Val Loss: 0.1700, Time: 0.66s
  [Supervised] Epoch 30/100 Train Loss: 0.1565, Val Loss: 0.1213, Time: 0.66s
  [Supervised] Epoch 35/100 Train Loss: 0.1201, Val Loss: 0.0983, Time: 0.66s
  [Supervised] Epoch 40/100 Train Loss: 0.0952, Val Loss: 0.1030, Time: 0.67s
  [Supervised] Epoch 45/100 Train Loss: 0.0676, Val Loss: 0.0560, Time: 0.67s
  [Supervised] Epoch 50/100 Train 

In [ ]:
!cp -r "/content/results_multiround_CNN1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D LeNet

In [ ]:
class LeNet1D(nn.Module):
    def __init__(self, in_channel=1, out_channel=4):
        super(LeNet1D, self).__init__()

        # --- Bloco Convolucional 1 ---
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channel, 6, kernel_size=64, stride=4, padding=30),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2), # 3000 -> 1500
        )

        # --- Bloco Convolucional 2 ---
        self.conv2 = nn.Sequential(
            nn.Conv1d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(12)
        )

        # --- Classificador (Fully Connected) ---
        # Input achatado: 16 canais * 12 pontos = 192 features
        self.fc1 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 12, 120), # Tamanho clássico da LeNet-5
            nn.ReLU()
        )

        self.fc2 = nn.Sequential(
            nn.Linear(120, 84), # Tamanho clássico da LeNet-5
            nn.ReLU()
        )

        self.fc3 = nn.Linear(84, out_channel)

    def forward(self, x):
        # Garante dimensão de canal (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        x = self.conv1(x)
        x = self.conv2(x)

        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração LeNet1D Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base LeNet (será copiado a cada iteração)
# Nota: Certifique-se de que a classe LeNet1D é a versão corrigida (LeNet1D_12k)
# que aceita in_channel e out_channel no construtor.
base_model = LeNet1D(in_channel=1, out_channel=num_classes)

# Listas para armazenar métricas
multiround_results_LeNet = []
accuracies_LeNet = []
f1_scores_LeNet = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para resetar os pesos do modelo
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"lenet1d_12k_round_{round_idx}",
        description=f"1D LeNet Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # LeNet é supervisionada, não tem pré-treino AE

        # Organização de saídas
        output_dir=f"results_multiround_LeNet1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_LeNet.append(result)

    # Coleta métricas globais desta rodada
    accuracies_LeNet.append(result.overall_metrics['accuracy'])
    # Tenta pegar mean_f1 ou f1, dependendo de como sua lib calcula
    f1_scores_LeNet.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL LENET MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_LeNet)
std_acc = np.std(accuracies_LeNet)
mean_f1 = np.mean(f1_scores_LeNet)
std_f1 = np.std(f1_scores_LeNet)

print(f"Rounds Executados: {len(multiround_results_LeNet)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração LeNet1D Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 1.3411, Val Loss: 1.3220, Time: 0.12s
  [Supervised] Epoch 5/100 Train Loss: 1.2004, Val Loss: 1.2243, Time: 0.11s
  [Supervised] Epoch 10/100 Train Loss: 1.0084, Val Loss: 1.0270, Time: 0.11s
  [Supervised] Epoch 15/100 Train Loss: 0.8577, Val Loss: 0.8847, Time: 0.13s
  [Supervised] Epoch 20/100 Train Loss: 0.7780, Val Loss: 0.8100, Time: 0.11s
  [Supervised] Epoch 25/100 Train Loss: 0.7071, Val Loss: 0.7497, Time: 0.11s
  [Supervised] Epoch 30/100 Train Loss: 0.6459, Val Loss: 0.7234, Time: 0.11s
  [Supervised] Epoch 35/100 Train Loss: 0.5857, Val Loss: 0.6671, Time: 0.11s
  [Supervised] Epoch 40/100 Train Loss: 0.5120, Val Loss: 0.5562, Time: 0.11s
  [Supervised] Epoch 45/100 Train Loss: 0.4540, Val Loss: 0.4775, Time: 0.11s
  [Supervised] Epoch 50/100 Tr

In [ ]:
!cp -r "/content/results_multiround_LeNet1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

### 1D ResNet18

In [ ]:
def conv3x1(in_planes, out_planes, stride=1):
    """3x1 convolution with padding"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x1(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x1(planes, planes)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet18(nn.Module):
    """
    Implementação da ResNet-18 adaptada para sinais 1D de 12.000 pontos.
    """
    def __init__(self, input_length=12000, in_channel=1, num_classes=10):
        super(ResNet18, self).__init__()

        # Configuração padrão da ResNet18
        block = BasicBlock
        layers = [2, 2, 2, 2] # 2 blocos por camada = 18 layers total

        self.inplanes = 64

        # STEM (Camada de Entrada)
        # Input: (Batch, 1, 12000)
        self.conv1 = nn.Conv1d(in_channel, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # Blocos Residuais
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # Classificador
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # Inicialização de Pesos
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm1d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        # Tratamento de Dimensão: Garante (Batch, 1, Length)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Head
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração ResNet18 Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base ResNet18
# Certifique-se de usar a classe adaptada para 1D (ResNet18_12k ou ResNet18_1D)
# que definimos anteriormente para aceitar o input_length corretamente.
base_model = ResNet18(input_length=input_length, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_ResNet = []
accuracies_ResNet = []
f1_scores_ResNet = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir que começamos com pesos limpos a cada rodada
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"resnet18_12k_round_{round_idx}",
        description=f"ResNet18 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # Nota: ResNet18 é pesada. Se der erro de memória (OOM), reduza para 32 ou 16.
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # ResNet é puramente supervisionada

        # Organização de saídas
        output_dir=f"results_multiround_ResNet18_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_ResNet.append(result)

    # Coleta métricas globais desta rodada
    accuracies_ResNet.append(result.overall_metrics['accuracy'])
    f1_scores_ResNet.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL RESNET18 MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_ResNet)
std_acc = np.std(accuracies_ResNet)
mean_f1 = np.mean(f1_scores_ResNet)
std_f1 = np.std(f1_scores_ResNet)

print(f"Rounds Executados: {len(multiround_results_ResNet)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração ResNet18 Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 0.9586, Val Loss: 1.8515, Time: 4.79s
  [Supervised] Epoch 5/100 Train Loss: 0.2742, Val Loss: 0.1271, Time: 3.43s
  [Supervised] Epoch 10/100 Train Loss: 0.1032, Val Loss: 0.0422, Time: 3.44s
  [Supervised] Epoch 15/100 Train Loss: 0.0740, Val Loss: 0.1147, Time: 3.45s
  [Supervised] Epoch 20/100 Train Loss: 0.0167, Val Loss: 0.0651, Time: 3.47s
  [Supervised] Epoch 25/100 Train Loss: 0.0104, Val Loss: 0.0128, Time: 3.48s
  [Supervised] Epoch 30/100 Train Loss: 0.0326, Val Loss: 0.0865, Time: 3.50s
  [Supervised] Epoch 35/100 Train Loss: 0.0055, Val Loss: 0.0151, Time: 3.51s
  [Supervised] Epoch 40/100 Train Loss: 0.0095, Val Loss: 0.0751, Time: 3.53s
  [Supervised] Epoch 45/100 Train Loss: 0.0114, Val Loss: 0.0273, Time: 3.54s
  [Supervised] Epoch 50/100 T

In [ ]:
!cp -r "/content/results_multiround_ResNet18_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

cp: cannot create directory '/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados': No such file or directory


### 1D AlexNet

In [ ]:
class AlexNet1D(nn.Module):
    def __init__(self, input_length=12000, in_channel=1, num_classes=10):
        super(AlexNet1D, self).__init__()

        self.features = nn.Sequential(
            # Conv1: Kernel 11 e Stride 4.
            # Reduz entrada 12000 -> ~3000
            nn.Conv1d(in_channel, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 3000 -> 1500

            # Conv2
            nn.Conv1d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 1500 -> 750

            # Conv3
            nn.Conv1d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv4
            nn.Conv1d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv5
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2), # 750 -> 375
        )

        self.avgpool = nn.AdaptiveAvgPool1d(6)

        self.classifier = nn.Sequential(
            nn.Dropout(),
            # 256 canais * 6 dimensão temporal = 1536 features
            nn.Linear(256 * 6, 1024), # Mantido 1024 conforme seu código (o paper original usa 4096)
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        # Tratamento de segurança para dimensão (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        x = self.features(x)
        x = self.avgpool(x)

        # Flatten robusto
        x = torch.flatten(x, 1)

        x = self.classifier(x)
        return x

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração AlexNet Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base AlexNet (será copiado a cada iteração)
# Garante que os parâmetros batem com a definição da sua classe AlexNet1D
base_model = AlexNet1D(input_length=input_length, in_channel=1, num_classes=num_classes)

# Listas para armazenar métricas
multiround_results_AlexNet = []
accuracies_AlexNet = []
f1_scores_AlexNet = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir que começamos com pesos limpos a cada rodada
    # Evita data leakage entre rounds
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"alexnet_12k_round_{round_idx}",
        description=f"AlexNet 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada (Array 1D)
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # AlexNet tem mais parâmetros que LeNet, mas menos que ResNet.
        # Batch 64 costuma ser seguro para 12k pontos em GPUs médias (8GB+ VRAM).
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0, # AlexNet é supervisionada padrão

        # Organização de saídas
        output_dir=f"results_multiround_AlexNet1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_AlexNet.append(result)

    # Coleta métricas globais desta rodada
    accuracies_AlexNet.append(result.overall_metrics['accuracy'])
    f1_scores_AlexNet.append(result.overall_metrics.get('mean_f1', result.overall_metrics.get('f1')))

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL ALEXNET MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_AlexNet)
std_acc = np.std(accuracies_AlexNet)
mean_f1 = np.mean(f1_scores_AlexNet)
std_f1 = np.std(f1_scores_AlexNet)

print(f"Rounds Executados: {len(multiround_results_AlexNet)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração AlexNet Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 1.3417, Val Loss: 1.2840, Time: 0.95s
  [Supervised] Epoch 5/100 Train Loss: 1.0440, Val Loss: 1.0767, Time: 0.91s
  [Supervised] Epoch 10/100 Train Loss: 0.6716, Val Loss: 0.7049, Time: 0.91s
  [Supervised] Epoch 15/100 Train Loss: 0.5239, Val Loss: 0.6223, Time: 0.92s
  [Supervised] Epoch 20/100 Train Loss: 0.5120, Val Loss: 0.5763, Time: 0.91s
  [Supervised] Epoch 25/100 Train Loss: 0.3866, Val Loss: 0.6663, Time: 0.91s
  [Supervised] Epoch 30/100 Train Loss: 0.2549, Val Loss: 0.3318, Time: 0.91s
  [Supervised] Epoch 35/100 Train Loss: 0.2535, Val Loss: 0.3066, Time: 0.91s
  [Supervised] Epoch 40/100 Train Loss: 0.2251, Val Loss: 0.2758, Time: 0.91s
  [Supervised] Epoch 45/100 Train Loss: 0.2180, Val Loss: 0.2906, Time: 0.91s
  [Supervised] Epoch 50/100 Tr

In [ ]:
!cp -r "/content/results_multiround_AlexNet1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

cp: cannot create directory '/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados': No such file or directory


### 1D BiLSTM

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, in_channel=1, out_channel=10):
        super(BiLSTM, self).__init__()

        # Hiperparâmetros Internos
        self.hidden_dim = 64
        self.kernel_num = 16
        self.num_layers = 2

        # self.V define o comprimento da sequência que entra na LSTM
        self.V = 100

        # Camada de stem
        self.embed1 = nn.Sequential(
            # Kernel grande e Stride 4 para lidar com alta taxa de amostragem
            nn.Conv1d(in_channel, self.kernel_num, kernel_size=64, stride=4, padding=30),
            nn.BatchNorm1d(self.kernel_num),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )

        self.embed2 = nn.Sequential(
            nn.Conv1d(self.kernel_num, self.kernel_num*2, kernel_size=3, padding=1),
            nn.BatchNorm1d(self.kernel_num*2),
            nn.ReLU(inplace=True),
            # AdaptiveMaxPool força a saída a ter exatamente comprimento self.V (100) garantindo entrada constante para a LSTM
            nn.AdaptiveMaxPool1d(self.V)
        )

        # Camada Recorrente (BiLSTM)
        # Input Size: kernel_num*2 (32 features por passo de tempo)
        self.bilstm = nn.LSTM(
            input_size=self.kernel_num*2,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            bidirectional=True,
            batch_first=True,
            bias=False
        )

        # Classificador
        # O input da linear é: Comprimento da Sequência (V) * (Hidden * 2 direções)
        self.hidden2label1 = nn.Sequential(
            nn.Linear(self.V * 2 * self.hidden_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.hidden2label2 = nn.Linear(512, out_channel)

    def forward(self, x):
        # (Batch, 1, 12000)
        if x.ndim == 2:
            x = x.unsqueeze(1)

        # Extração de Features Convolucionais
        x = self.embed1(x)
        x = self.embed2(x)
        # (Batch, 32, 100) -> (Batch, Channels, Time)

        # Preparação para LSTM
        # LSTM espera (Batch, Time, Features/Channels)
        x = x.permute(0, 2, 1) # (Batch, 100, 32)

        # Processamento Recorrente
        bilstm_out, _ = self.bilstm(x)
        bilstm_out = torch.tanh(bilstm_out)

        # Classificação
        bilstm_out = bilstm_out.reshape(bilstm_out.size(0), -1)

        logit = self.hidden2label1(bilstm_out)
        logit = self.hidden2label2(logit)

        return logit

In [ ]:
# 1. Configuração Inicial e Modelo Base
input_length = deep_dataset_time[0]['signal'][0].shape[-1]
num_classes = len(set([s['metainfo']['label'] for s in deep_dataset_time]))

print(f"--- Configuração BiLSTM Multiround ---")
print(f"Input length: {input_length}")
print(f"Num classes: {num_classes}")
# Assume que folds_multiround_deep já foi gerado
print(f"Total de Rounds: {len(folds_multiround_deep)}")

# Instancia o modelo base BiLSTM
# Certifique-se de que sua classe BiLSTM aceita input_length (se precisar para a FC final)
# ou se ela usa GlobalPooling para lidar com qualquer tamanho.
base_model = BiLSTM(in_channel=1, out_channel=num_classes)

# Listas para armazenar métricas
multiround_results_BiLSTM = []
accuracies_BiLSTM = []
f1_scores_BiLSTM = []

# 2. Loop de Execução Multiround
for round_idx, current_folds in enumerate(folds_multiround_deep):
    print(f"\n>>> Iniciando Round {round_idx + 1}/{len(folds_multiround_deep)} <<<")

    # Deepcopy para garantir pesos novos a cada rodada
    model_copy = copy.deepcopy(base_model)

    exp = DeepLearningExperiment(
        name=f"bilstm_12k_round_{round_idx}",
        description=f"BiLSTM 1D Multiround (Round {round_idx})",
        dataset=deep_dataset_time,

        # AQUI: Passamos os folds específicos desta rodada
        data_fold_idxs=current_folds,

        model=model_copy,

        # Hiperparâmetros de Treino
        # ATENÇÃO: LSTM com seq_len=12000 é muito pesado para a GPU.
        # Se der erro "CUDA Out of Memory", diminua batch_size para 32, 16 ou até 8.
        batch_size=64,
        lr=3e-4,
        num_epochs=100,
        pretrain_epochs=0,

        # Organização de saídas
        output_dir=f"results_multiround_BiLSTM1D_CWRU12K/round_{round_idx}"
    )

    # Executa o experimento
    result = exp.run()
    multiround_results_BiLSTM.append(result)

    # Coleta métricas globais desta rodada
    accuracies_BiLSTM.append(result.overall_metrics['accuracy'])
    f1_scores_BiLSTM.append(result.overall_metrics['mean_f1'])

    print(f"Round {round_idx + 1} Finalizado: Acc={result.overall_metrics['accuracy']:.4f}")

# 3. Relatório Final Consolidado
print("\n" + "="*40)
print("RELATÓRIO FINAL BILSTM MULTIROUND")
print("="*40)

mean_acc = np.mean(accuracies_BiLSTM)
std_acc = np.std(accuracies_BiLSTM)
mean_f1 = np.mean(f1_scores_BiLSTM)
std_f1 = np.std(f1_scores_BiLSTM)

print(f"Rounds Executados: {len(multiround_results_BiLSTM)}")
print(f"Acurácia Média:  {mean_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Score Médio:  {mean_f1:.4f} ± {std_f1:.4f}")
print("="*40)

--- Configuração BiLSTM Multiround ---
Input length: 12000
Num classes: 4
Total de Rounds: 8

>>> Iniciando Round 1/8 <<<

=== Outer Fold 1/4 ===
[Fold 0] Classifier training (100 epochs)...
  [Supervised] Epoch 1/100 Train Loss: 1.1772, Val Loss: 1.2515, Time: 0.43s
  [Supervised] Epoch 5/100 Train Loss: 0.6406, Val Loss: 0.5979, Time: 0.23s
  [Supervised] Epoch 10/100 Train Loss: 0.3936, Val Loss: 0.5996, Time: 0.24s
  [Supervised] Epoch 15/100 Train Loss: 0.3388, Val Loss: 0.3383, Time: 0.23s
  [Supervised] Epoch 20/100 Train Loss: 0.2504, Val Loss: 0.1614, Time: 0.23s
  [Supervised] Epoch 25/100 Train Loss: 0.1308, Val Loss: 0.1091, Time: 0.24s
  [Supervised] Epoch 30/100 Train Loss: 0.1471, Val Loss: 0.0749, Time: 0.24s
  [Supervised] Epoch 35/100 Train Loss: 0.1519, Val Loss: 0.1148, Time: 0.23s
  [Supervised] Epoch 40/100 Train Loss: 0.0598, Val Loss: 0.0832, Time: 0.24s
  [Supervised] Epoch 45/100 Train Loss: 0.0732, Val Loss: 0.3016, Time: 0.24s
  [Supervised] Epoch 50/100 Tra

In [ ]:
!cp -r "/content/results_multiround_BiLSTM1D_CWRU12K" "/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados"

cp: cannot create directory '/content/drive/MyDrive/Colab Notebooks/IFD-2025-2/Resultados': No such file or directory
